<a href="https://colab.research.google.com/github/Cheth-code/jugaad-ml/blob/main/Unsloth_TEXT_TO_SQl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.


In [ ]:
# Force-install the specific version that torchaudio wants
!pip install torch==2.9.0 torchaudio==2.9.0 --extra-index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 55.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 79.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 53.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 54.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 64.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 67.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 146.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 68.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 243.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # a higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.1.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("b-mc2/sql-create-context", split = "train")

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
print(dataset[:5])

{'answer': ['SELECT COUNT(*) FROM head WHERE age > 56', 'SELECT name, born_state, age FROM head ORDER BY age', 'SELECT creation, name, budget_in_billions FROM department', 'SELECT MAX(budget_in_billions), MIN(budget_in_billions) FROM department', 'SELECT AVG(num_employees) FROM department WHERE ranking BETWEEN 10 AND 15'], 'question': ['How many heads of the departments are older than 56 ?', 'List the name, born state and age of the heads of departments ordered by age.', 'List the creation year, name and budget of each department.', 'What are the maximum and minimum budget of the departments?', 'What is the average number of employees of the departments whose rank is between 10 and 15?'], 'context': ['CREATE TABLE head (age INTEGER)', 'CREATE TABLE head (name VARCHAR, born_state VARCHAR, age VARCHAR)', 'CREATE TABLE department (creation VARCHAR, name VARCHAR, budget_in_billions VARCHAR)', 'CREATE TABLE department (budget_in_billions INTEGER)', 'CREATE TABLE department (num_employees IN

In [ ]:

sql_r1_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a SQL expert. First, analyze the schema and plan your query inside <think> tags. Then, provide the final SQL query in the solution block.<|eot_id|><|start_header_id|>user<|end_header_id|>
### Database Schema:
{}

### Question:
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
<think>
{}
</think>

**Solution:**
```sql
{}
```<|eot_id|>"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
  schemas   = examples["context"]
  questions = examples["question"]
  solutions = examples["answer"]
  texts = []

  for schema, question, solution in zip(schemas, questions, solutions):
        thought = f"I need to query the table based on the question: '{question}'. Looking at the schema, I will use the relevant columns to form a SELECT statement."

        text = sql_r1_prompt.format(schema, question, thought, solution) + EOS_TOKEN
        texts.append(text)

  return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 10, # Few updates with low learning rate before actual training
        max_steps = 100, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 78,577 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
1,3.138500
2,3.236900
3,3.125600
4,3.204200
5,3.055800
6,3.139400
7,3.091000
8,2.789900
9,2.713800
10,2.679700


In [ ]:
from unsloth.chat_templates import get_chat_template

# 1. First Principles Fix: Define the SQL System Prompt clearly
sql_sys_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a SQL expert. Follow these three parsimonious rules exactly:
1. First, analyze the database schema and plan your logic inside <think> tags.
2. Provide the final SQL query in a markdown code block.
3. MANDATORY: You MUST end every SQL query with a semicolon (;). Failure to include the semicolon is an error.

### Schema:
{}<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# 2. Test Data: Use your specific Cloud Billing schema
test_schema = "CREATE TABLE gcp_billing (id INT, project_id TEXT, cost FLOAT, currency TEXT, usage_start TIMESTAMP,usage_end TIMESTAMP);"
test_question = "Find the top 3 services by cost for 'finance-01' in Dec 2025 and their % of total project spend."

# 3. Formatting: Use the correct variable 'sql_sys_prompt'
message = sql_sys_prompt.format(test_schema, test_question)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": message},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Optimizer Fix: Lower temperature for SQL precision
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,  # Deterministic output for SQL
    min_p = 0.1
)

input_length = inputs.shape[1]
clean_output_tokens = outputs[0][input_length:]
clean_response = tokenizer.decode(clean_output_tokens, skip_special_tokens=True)
print(clean_response.split("assistant")[-1].strip())

<think>
I will query the table based on the schema provided. I need to find the top 3 services by cost for 'finance-01' in Dec 2025 and their % of total project spend. I will use the relevant columns from the table: project_id, cost, and usage_start. I will filter the data by project_id and usage_start, then group by project_id and service (which I assume is the service name), and finally use the ROW_NUMBER() function to get the top 3 services.
</think>

```sql
SELECT 
  project_id,
  service,
  cost,
  (cost / SUM(cost) OVER (PARTITION BY project_id)) * 100 AS percentage_of_total_project_spend
FROM (
  SELECT 
    project_id,
    id,
    cost,
    currency,
    usage_start,
    usage_end,
    ROW_NUMBER() OVER (PARTITION BY project_id ORDER BY cost DESC) AS row_num,
    CASE
      WHEN project_id = 'finance-01' AND EXTRACT(YEAR FROM usage_start) = 2025 AND EXTRACT(MONTH FROM usage_start) = 12 THEN id
    END AS service
  FROM gcp_billing
) AS subquery
WHERE row_num <= 3;
```


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# The final 'Optimizer' move to make your model public
model.push_to_hub("Chethan102/Text_sql_Ai-Cloud-Billing", token = True)

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  559kB /  336MB            

Saved model to https://huggingface.co/Chethan102/Text_sql_Ai-Cloud-Billing
